<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/02_filtering_where.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Filtering rows (WHERE)

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~30 min**

## Learning objectives
- filter rows with `WHERE` and comparisons (`=`, `<`, `>`, `<>`);
- combine conditions with `AND`, `OR`, `NOT`;
- use `IN`, `BETWEEN`, `LIKE` and `IS NULL`.

`WHERE` keeps only the rows that match a condition. It is the workhorse of every query.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## Setup — run this cell first

This connects the notebook to the example database. It works both on your own computer and on Google Colab; just run it (Shift+Enter). You do not need to understand it.

In [1]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.6/397.6 kB 18.3 MB/s eta 0:00:00


Connecting to 'sqlite:///../data/omics.db'

Connected to omics.db — you are ready.


## 1. A simple condition

In [3]:
%%sql
SELECT * FROM samples WHERE environment = 'Soil'


Running query in 'sqlite:///../data/omics.db'

sample_id,site,environment,treatment,ph,temperature_c,depth_cm,collection_date
S001,LEI_SOIL,Soil,Control,5.88,14.5,9.3,2025-04-11
S002,LEI_SOIL,Soil,Control,6.46,13.4,18.1,2025-08-05
S003,LEI_SOIL,Soil,Control,6.09,13.5,19.3,2025-08-25
S004,LEI_SOIL,Soil,Amended,6.17,17.8,None,2025-07-23
S005,LEI_SOIL,Soil,Amended,6.04,19.1,5.6,2025-04-07
S006,LEI_SOIL,Soil,Amended,6.01,19.3,13.3,2025-08-01
S007,UPP_SOIL,Soil,Control,6.46,17.6,3.9,2025-06-17
S008,UPP_SOIL,Soil,Control,5.71,14.3,4.2,2025-09-24
S009,UPP_SOIL,Soil,Control,None,16.7,6.9,2025-06-26
S010,UPP_SOIL,Soil,Amended,5.85,19.4,18.0,2025-04-09


The opposite test uses `<>` (not equal). Here, every sample that is not Soil.

In [4]:
%%sql
-- <> means 'not equal to'
SELECT sample_id, environment
FROM samples
WHERE environment <> 'Soil';


Running query in 'sqlite:///../data/omics.db'

sample_id,environment
S013,Freshwater
S014,Freshwater
S015,Freshwater
S016,Freshwater
S017,Freshwater
S018,Freshwater
S019,Freshwater
S020,Freshwater
S021,Freshwater
S022,Freshwater


## 2. Numeric comparison

In [16]:
%%sql
SELECT count(*) FROM samples WHERE ph=6.5;

Running query in 'sqlite:///../data/omics.db'

count(*)
0


## 3. Combine with AND / OR

In [10]:
%%sql
SELECT sample_id, environment, treatment
FROM samples
WHERE environment = 'Freshwater' AND treatment = 'Amended';

Running query in 'sqlite:///../data/omics.db'

sample_id,environment,treatment
S016,Freshwater,Amended
S017,Freshwater,Amended
S018,Freshwater,Amended
S022,Freshwater,Amended
S023,Freshwater,Amended
S024,Freshwater,Amended


In [21]:
%%sql
--did it myself , selct is laways followed by from
SELECT sample_id, Treatment, ph
FROM samples
WHERE TREATMENT = "Amended"
LIMIT 10



Running query in 'sqlite:///../data/omics.db'

sample_id,treatment,ph
S004,Amended,6.17
S005,Amended,6.04
S006,Amended,6.01
S010,Amended,5.85
S011,Amended,6.4
S012,Amended,5.87
S016,Amended,7.69
S017,Amended,7.43
S018,Amended,7.17
S022,Amended,7.55


`OR` widens the net; wrap `OR` in parentheses when you mix it with `AND` so the logic is unambiguous.

In [27]:
%%sql
-- amended samples that are EITHER acidic OR warm
SELECT sample_id, environment, ph, temperature_c
FROM samples
WHERE treatment = 'Amended'
  AND (ph < 6.5 OR temperature_c > 18);

Running query in 'sqlite:///../data/omics.db'

sample_id,environment,ph,temperature_c
S004,Soil,6.17,17.8
S005,Soil,6.04,19.1
S006,Soil,6.01,19.3
S010,Soil,5.85,19.4
S011,Soil,6.4,19.8
S012,Soil,5.87,20.9
S016,Freshwater,7.69,24.6
S017,Freshwater,7.43,20.0
S018,Freshwater,7.17,20.2
S022,Freshwater,7.55,25.1


## 4. IN — match a list of values

In [ ]:
%%sql
SELECT genus, phylum FROM taxa
WHERE phylum IN ('Cyanobacteriota', 'Euryarchaeota');

In [28]:
%%sql
---did it myself
SELECT Genus, phylum FROM taxa WHERE PHYLUM = "Cyanobacteriota" OR "Euryarchaeota"

Running query in 'sqlite:///../data/omics.db'

genus,phylum
Microcystis,Cyanobacteriota
Microcystis,Cyanobacteriota
Synechococcus,Cyanobacteriota
Synechococcus,Cyanobacteriota
Microcystis,Cyanobacteriota


`NOT IN` keeps everything except a list, useful to exclude a few known groups.

In [29]:
%%sql
-- all taxa EXCEPT the two archaeal/photosynthetic phyla above
SELECT genus, phylum
FROM taxa
WHERE phylum NOT IN ('Cyanobacteriota', 'Euryarchaeota')
LIMIT 10;

Running query in 'sqlite:///../data/omics.db'

genus,phylum
Nitrosomonas,Pseudomonadota
Pseudomonas,Pseudomonadota
Pseudomonas,Pseudomonadota
Nitrosomonas,Pseudomonadota
Escherichia,Pseudomonadota
Pseudomonas,Pseudomonadota
Chryseobacterium,Bacteroidota
Bacteroides,Bacteroidota
Bacteroides,Bacteroidota
Flavobacterium,Bacteroidota


## 5. BETWEEN — an inclusive range

In [30]:
%%sql
SELECT sample_id, ph FROM samples WHERE ph BETWEEN 6.5 AND 7.5;

Running query in 'sqlite:///../data/omics.db'

sample_id,ph
S014,7.37
S017,7.43
S018,7.17
S020,7.39
S023,7.41
S024,7.23
S025,7.27
S027,7.1
S028,6.99
S029,7.08


In [34]:
%%sql
-- did it myself i want to know what i am lookig aat
SELECT * FROM samples

Running query in 'sqlite:///../data/omics.db'

sample_id,site,environment,treatment,ph,temperature_c,depth_cm,collection_date
S001,LEI_SOIL,Soil,Control,5.88,14.5,9.3,2025-04-11
S002,LEI_SOIL,Soil,Control,6.46,13.4,18.1,2025-08-05
S003,LEI_SOIL,Soil,Control,6.09,13.5,19.3,2025-08-25
S004,LEI_SOIL,Soil,Amended,6.17,17.8,None,2025-07-23
S005,LEI_SOIL,Soil,Amended,6.04,19.1,5.6,2025-04-07
S006,LEI_SOIL,Soil,Amended,6.01,19.3,13.3,2025-08-01
S007,UPP_SOIL,Soil,Control,6.46,17.6,3.9,2025-06-17
S008,UPP_SOIL,Soil,Control,5.71,14.3,4.2,2025-09-24
S009,UPP_SOIL,Soil,Control,None,16.7,6.9,2025-06-26
S010,UPP_SOIL,Soil,Amended,5.85,19.4,18.0,2025-04-09


## 6. LIKE — text pattern (`%` = any characters)

In [35]:
%%sql
SELECT genus, phylum FROM taxa WHERE genus LIKE 'B%';

Running query in 'sqlite:///../data/omics.db'

genus,phylum
Bacteroides,Bacteroidota
Bacteroides,Bacteroidota
Bacillus,Bacillota


In [40]:
%%sql
--i dit it myself, like, not in and simillar have ""
SELECT genus, phylum FROM taxa WHERE genus LIKE '%us' AND phylum LIKE "%a";
SELECT genus, phylum from taxa where genus = "Bacteroides"

Running query in 'sqlite:///../data/omics.db'

genus,phylum
Bacteroides,Bacteroidota
Bacteroides,Bacteroidota


`%` matches any run of characters; `_` matches exactly one. You can also anchor the pattern at the end.

In [41]:
%%sql
-- families ending in 'aceae' (the standard bacterial family suffix)
SELECT DISTINCT family
FROM taxa
WHERE family LIKE '%aceae';

Running query in 'sqlite:///../data/omics.db'

family
Pseudomonadaceae
Bacteroidaceae
Bacillaceae
Mycobacteriaceae
Acidobacteriaceae
Synechococcaceae
Methanosarcinaceae
Nitrospiraceae


## 7. IS NULL — missing values
Some samples have no recorded pH. You must use `IS NULL`, never `= NULL`.

In [ ]:
%%sql
SELECT sample_id, environment, ph FROM samples WHERE ph IS NULL;

`IS NOT NULL` is the mirror image: keep only rows that do have a value. Combine it with another condition freely.

In [ ]:
%%sql
-- samples that HAVE a pH and are on the acidic side
SELECT sample_id, ph
FROM samples
WHERE ph IS NOT NULL AND ph < 6.5;

---
## Exercises

**Exercise.** Show all samples from the `Amended` treatment.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT * FROM samples WHERE treatment = 'Amended';
```
</details>

**Exercise.** List the genus and family of all taxa in the phylum `Bacillota`.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT genus, family FROM taxa WHERE phylum = 'Bacillota';
```
</details>

**Exercise.** Find samples warmer than 20 °C (`temperature_c`).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id, temperature_c FROM samples WHERE temperature_c > 20;
```
</details>

**Exercise.** Find the samples whose `depth_cm` is missing.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id FROM samples WHERE depth_cm IS NULL;
```
</details>

### Recap
- `WHERE condition` keeps matching rows; combine with `AND`/`OR`/`NOT`.
- `IN (…)`, `BETWEEN a AND b`, `LIKE 'B%'` are shortcuts.
- Missing values need `IS NULL` / `IS NOT NULL` (never `= NULL`).
- Next: 03 · Sorting, limiting & computed columns.